# Load One AICME Model And Export All Prediction/VPC Plots

This notebook keeps only the minimal workflow needed for inspection:

1. Select one trained experiment folder.
2. Load the checkpoint and empirical datamodule.
3. Build the list of every available empirical dataset, batch, study, and drug.
4. Run prediction plots separately from VPC plots so the slow VPC export can be skipped.
5. Run one synthetic prediction example and save it under the same output tree.


In [1]:
from __future__ import annotations

import re
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch


def find_project_root(start: Path) -> Path:
    """Return the repository root used by this notebook."""

    for candidate in [start, *start.parents]:
        if (candidate / "pff").exists() and (candidate / "config_files").exists():
            return candidate
    raise RuntimeError("Could not find the project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pff.config_classes.node_pk_config import NodePKExperimentConfig
from pff.data.data_empirical.builder import (
    databatch_to_study_jsons,
    prediction_to_study_jsons,
)
from pff.data.datasets.aicme_datasets import AICMECompartmentsDataModule
from pff.metrics.sampling_quality import compute_vpc_data, vpc_plot
from pff.models.amortized_inference.aicme import AICMEPK
from pff.training.utils import (
    get_lightning_checkpoint_path,
    load_model_from_checkpoint_path,
)
from pff.utils.plots.databatch_plot import plot_study_json_with_prediction

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


def discover_experiment_dirs(results_dir: Path) -> list[Path]:
    """Return available experiment folders ordered by modification time."""

    if not results_dir.exists():
        return []
    candidates = [
        path
        for path in results_dir.iterdir()
        if path.is_dir() and (path / "experiment_config.yaml").exists()
    ]
    return sorted(candidates, key=lambda path: path.stat().st_mtime, reverse=True)


def normalize_name(name: object) -> str:
    """Normalize free-text labels for robust comparisons."""

    return "".join(ch.lower() for ch in str(name) if ch.isalnum())


def safe_name(name: object) -> str:
    """Convert labels into filesystem-safe path components."""

    text = str(name).strip().replace("/", "_")
    text = re.sub(r"[^0-9A-Za-z._-]+", "-", text)
    text = re.sub(r"-+", "-", text).strip("-._")
    return text or "unknown"


def split_dataset_key(dataset_key: str) -> tuple[str, str]:
    """Split `user/dataset` keys while tolerating unexpected formats."""

    parts = str(dataset_key).split("/", maxsplit=1)
    if len(parts) == 2:
        return parts[0], parts[1]
    return "unknown_user", parts[0]


RESULTS_DIR = PROJECT_ROOT / "results" / "comet" / "aistats"
OUTPUT_ROOT = PROJECT_ROOT / "reports" / "results_aistats"
AVAILABLE_EXPERIMENT_DIRS = discover_experiment_dirs(RESULTS_DIR)

# Select the experiment you want to inspect here.
EXPERIMENT_DIR = AVAILABLE_EXPERIMENT_DIRS[0] if AVAILABLE_EXPERIMENT_DIRS else None
EXPERIMENT_DIR = Path("/home/cesarali/Pharma/pff/results/comet/uai/cb906bbb30f34522813c1357fff45371")
CHECKPOINT_TYPE = "best"  # "best" or "last"
MAP_LOCATION = "cpu"  # "cpu" or "cuda"
FIX_PAST_TARGET_STEPS = 5  # Default fixed number of past target steps
FORCE_LEGACY_LATENT_LAYOUT = True  # Keep old checkpoints loadable after latent split

PREDICTION_SAMPLE_SIZE = 8
PREDICTION_LOG_SCALE = True
VPC_SAMPLE_SIZE = 200
VPC_N_BINS = 10
VPC_BINNING = "equal_count"
VPC_LOG_SCALE = False

RUN_SYNTHETIC_EXAMPLE = True
SYNTHETIC_SPLIT = "test"
SYNTHETIC_BATCH_INDEX = 0
SYNTHETIC_PERMUTATION_INDEX = 0

predictive_plot_kwargs = {
    "figure_size": (8, 6),
    "title": None,
    "title_font_size": 19,
    "show_legend": True,
    "legend_font_size": 12,
    "axis_label_font_size": 18,
    "tick_label_font_size": 14,
    "point_size": 32,
    "point_marker": "o",
    "prediction_marker": "o",
    "prediction_marker_size": 9,
    "context_obs_color": "forestgreen",
    "context_rem_color": "mediumseagreen",
}

if EXPERIMENT_DIR is None:
    raise FileNotFoundError(
        f"No experiment folders with experiment_config.yaml were found under {RESULTS_DIR}."
    )


def force_legacy_aicme_latent_layout(exp_config: NodePKExperimentConfig) -> NodePKExperimentConfig:
    """Force AICME to use the pre-split latent layout for legacy checkpoints."""

    base_latent_dim = int(exp_config.network.zi_latent_dim)
    exp_config.network = replace(
        exp_config.network,
        z_s_latent_dim=base_latent_dim,
        z_i_latent_dim=base_latent_dim,
    )
    return exp_config


def initialize_identity_latent_adapters(model: AICMEPK) -> None:
    """Make newly introduced decoder adapters behave like the old architecture."""

    with torch.no_grad():
        for layer_name in ("study_to_decoder_latent", "individual_to_decoder_latent"):
            layer = getattr(model, layer_name, None)
            if layer is None:
                continue
            out_dim, in_dim = layer.weight.shape
            if in_dim != out_dim:
                continue
            layer.weight.copy_(
                torch.eye(out_dim, device=layer.weight.device, dtype=layer.weight.dtype)
            )
            layer.bias.zero_()


def load_aicme_experiment(
    experiment_dir: Path,
    *,
    checkpoint_type: str = "best",
    map_location: str = "cpu",
    force_legacy_latent_layout: bool = False,
) -> tuple[NodePKExperimentConfig, AICMEPK, AICMECompartmentsDataModule, Path]:
    """Load one AICME experiment together with its empirical datamodule."""

    config_path = experiment_dir / "experiment_config.yaml"
    if not config_path.exists():
        raise FileNotFoundError(f"Missing experiment config: {config_path}")

    exp_config = NodePKExperimentConfig.from_yaml(str(config_path))
    checkpoint_path = get_lightning_checkpoint_path(str(experiment_dir), checkpoint_type)
    if checkpoint_path is None:
        raise FileNotFoundError(
            f"Could not find a '{checkpoint_type}' checkpoint inside {experiment_dir}."
        )
    if force_legacy_latent_layout:
        exp_config = force_legacy_aicme_latent_layout(exp_config)

    device = torch.device(map_location)
    model = load_model_from_checkpoint_path(
        AICMEPK,
        checkpoint_path,
        model_config=exp_config,
        map_location=map_location,
        strict=not force_legacy_latent_layout,
    )
    if force_legacy_latent_layout:
        initialize_identity_latent_adapters(model)
    model = model.to(device)
    model.eval()

    datamodule = AICMECompartmentsDataModule(exp_config)
    datamodule.setup()
    return exp_config, model, datamodule, Path(checkpoint_path)


exp_config, model, datamodule, checkpoint_path = load_aicme_experiment(
    EXPERIMENT_DIR,
    checkpoint_type=CHECKPOINT_TYPE,
    map_location=MAP_LOCATION,
    force_legacy_latent_layout=FORCE_LEGACY_LATENT_LAYOUT,
)

if FIX_PAST_TARGET_STEPS is not None:
    datamodule.fix_past_selection(FIX_PAST_TARGET_STEPS, who="target")
else:
    datamodule.release_past_selection(who="target")

print("Using experiment folder:", EXPERIMENT_DIR)
print("Checkpoint:", checkpoint_path)
print("Model:", type(model).__name__)
print("Configured empirical datasets:", exp_config.mix_data.test_empirical_datasets)
print("Output root:", OUTPUT_ROOT)
AVAILABLE_EXPERIMENT_DIRS[:10]


RuntimeError: Error(s) in loading state_dict for AICMEPK:
	size mismatch for mu_s_layer.weight: copying a param with shape torch.Size([64, 128]) from checkpoint, the shape in current model is torch.Size([128, 128]).
	size mismatch for mu_s_layer.bias: copying a param with shape torch.Size([64]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for logvar_s_layer.weight: copying a param with shape torch.Size([64, 128]) from checkpoint, the shape in current model is torch.Size([128, 128]).
	size mismatch for logvar_s_layer.bias: copying a param with shape torch.Size([64]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for study_to_decoder_latent.weight: copying a param with shape torch.Size([128, 64]) from checkpoint, the shape in current model is torch.Size([128, 128]).

In [6]:
def iter_empirical_selections(datamodule: AICMECompartmentsDataModule):
    """Yield one plotting selection per dataset, batch, study, and drug."""

    heldout_map = datamodule.get_empirical_test_batches(no_heldout=False)
    no_heldout_map = datamodule.get_empirical_test_batches(no_heldout=True)

    for dataset_key, heldout_batches in heldout_map.items():
        paired_vpc_batches = no_heldout_map.get(dataset_key, [])
        for batch_index, heldout_batch in enumerate(heldout_batches):
            studies, drugs = datamodule.describe_empirical_batch(
                heldout_batch,
                print_available=False,
            )

            for substance_index, drug_name in enumerate(drugs):
                study_name = (
                    studies[substance_index]
                    if substance_index < len(studies)
                    else f"study_{substance_index:03d}"
                )
                heldout_single = datamodule.slice_single_substance_batch(
                    heldout_batch,
                    substance_index,
                )

                no_heldout_single = None
                if batch_index < len(paired_vpc_batches):
                    candidate_batch = paired_vpc_batches[batch_index]
                    try:
                        candidate_single = datamodule.slice_single_substance_batch(
                            candidate_batch,
                            substance_index,
                        )
                        candidate_drug = candidate_single.substance_name[0]
                        if normalize_name(candidate_drug) == normalize_name(drug_name):
                            no_heldout_single = candidate_single
                        else:
                            no_heldout_single = datamodule.slice_single_substance_batch_by_name(
                                candidate_batch,
                                drug_name,
                            )
                    except Exception:
                        try:
                            no_heldout_single = datamodule.slice_single_substance_batch_by_name(
                                candidate_batch,
                                drug_name,
                            )
                        except Exception:
                            no_heldout_single = None

                yield {
                    "dataset_key": dataset_key,
                    "batch_index": batch_index,
                    "study_name": study_name,
                    "drug_name": drug_name,
                    "heldout_batch": heldout_single,
                    "no_heldout_batch": no_heldout_single,
                }


def selection_output_dir(selection: dict[str, object]) -> Path:
    """Build one output folder per drug within each dataset."""

    user_name, dataset_name = split_dataset_key(str(selection["dataset_key"]))
    return (
        OUTPUT_ROOT
        / safe_name(EXPERIMENT_DIR.name)
        / safe_name(user_name)
        / safe_name(dataset_name)
        / safe_name(selection["drug_name"])
    )


def selection_file_stem(selection: dict[str, object]) -> str:
    """Build a unique filename stem for one batch/study/drug selection."""

    return (
        f"batch_{int(selection['batch_index']):03d}"
        f"__{safe_name(selection['study_name'])}"
        f"__{safe_name(selection['drug_name'])}"
    )


def synthetic_example_output_dir() -> Path:
    """Return the folder used for exported synthetic prediction examples."""

    return OUTPUT_ROOT / safe_name(EXPERIMENT_DIR.name) / "synthetic_examples"


def load_synthetic_prediction_batch(
    datamodule: AICMECompartmentsDataModule,
    *,
    split: str,
    batch_index: int,
    permutation_index: int,
):
    """Load one synthetic batch from the requested dataloader split."""

    split_name = str(split).strip().lower()
    if split_name == "train":
        loader = datamodule.train_dataloader()
    elif split_name == "val":
        loader = datamodule.val_dataloader()
    elif split_name == "test":
        loader = datamodule.test_dataloader()
    else:
        raise ValueError(
            f"Unsupported synthetic split '{split}'. Expected one of: train, val, test."
        )

    for current_batch_index, batch_list in enumerate(loader):
        if current_batch_index != int(batch_index):
            continue
        if not isinstance(batch_list, list):
            raise TypeError("Synthetic dataloader must return a list of permutation batches.")
        if permutation_index < 0 or permutation_index >= len(batch_list):
            raise IndexError(
                f"Synthetic permutation index {permutation_index} is out of range for batch "
                f"{batch_index} with {len(batch_list)} permutation(s)."
            )
        return batch_list[permutation_index].to("cpu")

    raise IndexError(
        f"Synthetic batch index {batch_index} is out of range for split '{split_name}'."
    )


def save_prediction_plot(
    *,
    model: AICMEPK,
    batch,
    image_path: Path,
    device: torch.device,
    sample_size: int,
    log_scale: bool,
    drug_name: str,
    plot_kwargs: dict[str, object],
) -> None:
    """Save the predictive plot for one held-out empirical selection."""

    batch_device = batch.to(device)
    batch_cpu = batch.to("cpu")

    with torch.inference_mode():
        (
            prediction_samples,
            prediction_times,
            _target_future,
            _target_future_mask,
        ) = model.sample_individual_prediction(
            batch_device,
            sample_size=sample_size,
        )

    # prediction_samples : [S, B, It, Tr, 1]
    # prediction_times   : [S, B, It, Tr, 1]
    studies_with_predictions = prediction_to_study_jsons(
        prediction_samples.detach().cpu(),
        prediction_times.detach().cpu(),
        batch_cpu,
        model.meta_dosing,
    )
    study = studies_with_predictions[0]

    render_kwargs = dict(plot_kwargs)
    figure_size = tuple(render_kwargs.pop("figure_size", (8, 6)))
    title = render_kwargs.pop("title", None) or drug_name.capitalize()
    title_font_size = render_kwargs.pop("title_font_size", None)

    fig, ax = plt.subplots(figsize=figure_size)
    plot_study_json_with_prediction(
        study,
        ax=ax,
        log_scale=log_scale,
        **render_kwargs,
    )
    if title_font_size is not None:
        ax.set_title(title, fontsize=float(title_font_size))
    else:
        ax.set_title(title)
    image_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(image_path, bbox_inches="tight")
    plt.close(fig)


def save_vpc_plot(
    *,
    model: AICMEPK,
    batch,
    image_path: Path,
    device: torch.device,
    sample_size: int,
    n_bins: int,
    binning: str,
    log_scale: bool,
) -> None:
    """Save the empirical VPC plot for one no-heldout selection."""

    batch_device = batch.to(device)
    batch_cpu = batch.to("cpu")
    observed_studies = databatch_to_study_jsons(batch_cpu, model.meta_dosing)

    with torch.inference_mode():
        simulated_studies_by_substance = model.sample_new_individuals_to_vpc_format(
            batch_device,
            sample_size=sample_size,
        )

    observed_study = observed_studies[0]
    simulated_studies = simulated_studies_by_substance[0]
    vpc_results = compute_vpc_data(
        observed_study,
        simulated_studies,
        n_bins=n_bins,
        binning=binning,
    )

    fig, ax = plt.subplots(figsize=(6, 4))
    vpc_plot(vpc_results, ax=ax, log_y=log_scale)
    ax.set_title(
        f"VPC: {observed_study['meta_data']['study_name']} | "
        f"{observed_study['meta_data']['substance_name']}"
    )
    image_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(image_path, bbox_inches="tight")
    plt.close(fig)


device = torch.device(MAP_LOCATION)
selections = list(iter_empirical_selections(datamodule))
print(f"Found {len(selections)} empirical selections.")

selections_summary = pd.DataFrame(
    [
        {
            "dataset_key": selection["dataset_key"],
            "batch_index": selection["batch_index"],
            "study_name": selection["study_name"],
            "drug_name": selection["drug_name"],
            "has_vpc_batch": selection["no_heldout_batch"] is not None,
        }
        for selection in selections
    ]
)



Found 198 empirical selections.


In [7]:
prediction_records: list[dict[str, object]] = []
for index, selection in enumerate(selections, start=1):
    output_dir = selection_output_dir(selection)
    prediction_path = output_dir / f"{selection_file_stem(selection)}__prediction.png"

    print(
        f"[prediction {index:03d}/{len(selections):03d}] "
        f"{selection['dataset_key']} | batch={selection['batch_index']} | "
        f"study={selection['study_name']} | drug={selection['drug_name']}"
    )

    prediction_status = "saved"
    prediction_saved = True
    try:
        save_prediction_plot(
            model=model,
            batch=selection["heldout_batch"],
            image_path=prediction_path,
            device=device,
            sample_size=PREDICTION_SAMPLE_SIZE,
            log_scale=PREDICTION_LOG_SCALE,
            drug_name=str(selection["drug_name"]),
            plot_kwargs=predictive_plot_kwargs,
        )
    except Exception as exc:
        prediction_status = f"failed: {exc}"
        prediction_saved = False
        prediction_path = None

    prediction_records.append(
        {
            "dataset_key": selection["dataset_key"],
            "batch_index": selection["batch_index"],
            "study_name": selection["study_name"],
            "drug_name": selection["drug_name"],
            "prediction_saved": prediction_saved,
            "prediction_status": prediction_status,
            "prediction_path": str(prediction_path) if prediction_path is not None else None,
        }
    )

prediction_summary = pd.DataFrame.from_records(prediction_records)
prediction_summary = prediction_summary.sort_values(
    by=["dataset_key", "batch_index", "study_name", "drug_name"]
).reset_index(drop=True)

print(f"Saved {int(prediction_summary['prediction_saved'].sum())} prediction plot(s).")
#prediction_summary


[prediction 001/198] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=memantine
[prediction 002/198] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=omeprazole
[prediction 003/198] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=5-hydroxyomeprazole
[prediction 004/198] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=omeprazole sulfone
[prediction 005/198] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=repaglinide
[prediction 006/198] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=hydroxy repaglinide
[prediction 007/198] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=rosuvastatin
[prediction 008/198] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=tolbutamide
[prediction 009/198] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=4-hydroxytolbutamide
[prediction 010/198] cesarali/lenuzza-2016 | batch=0 | study=Lenuzza2016 | drug=dextromethorphan
[prediction 011/198] cesarali/lenuzza-2016 | 

In [ ]:
vpc_records: list[dict[str, object]] = []
for index, selection in enumerate(selections, start=1):
    output_dir = selection_output_dir(selection)
    vpc_path = output_dir / f"{selection_file_stem(selection)}__vpc.png"

    print(
        f"[vpc {index:03d}/{len(selections):03d}] "
        f"{selection['dataset_key']} | batch={selection['batch_index']} | "
        f"study={selection['study_name']} | drug={selection['drug_name']}"
    )

    vpc_status = "saved"
    vpc_saved = True
    if selection["no_heldout_batch"] is None:
        vpc_status = "skipped: no matching no-heldout batch"
        vpc_saved = False
        vpc_path = None
    else:
        try:
            save_vpc_plot(
                model=model,
                batch=selection["no_heldout_batch"],
                image_path=vpc_path,
                device=device,
                sample_size=VPC_SAMPLE_SIZE,
                n_bins=VPC_N_BINS,
                binning=VPC_BINNING,
                log_scale=VPC_LOG_SCALE,
            )
        except Exception as exc:
            vpc_status = f"failed: {exc}"
            vpc_saved = False
            vpc_path = None

    vpc_records.append(
        {
            "dataset_key": selection["dataset_key"],
            "batch_index": selection["batch_index"],
            "study_name": selection["study_name"],
            "drug_name": selection["drug_name"],
            "vpc_saved": vpc_saved,
            "vpc_status": vpc_status,
            "vpc_path": str(vpc_path) if vpc_path is not None else None,
        }
    )

vpc_summary = pd.DataFrame.from_records(vpc_records)
vpc_summary = vpc_summary.sort_values(
    by=["dataset_key", "batch_index", "study_name", "drug_name"]
).reset_index(drop=True)

print(f"Saved {int(vpc_summary['vpc_saved'].sum())} VPC plot(s).")
vpc_summary


In [ ]:
synthetic_prediction_summary = pd.DataFrame(
    columns=[
        "split",
        "batch_index",
        "permutation_index",
        "drug_name",
        "prediction_saved",
        "prediction_status",
        "prediction_path",
    ]
)

if RUN_SYNTHETIC_EXAMPLE:
    synthetic_prediction_path = (
        synthetic_example_output_dir()
        / (
            f"{safe_name(SYNTHETIC_SPLIT)}_batch_{int(SYNTHETIC_BATCH_INDEX):03d}"
            f"__perm_{int(SYNTHETIC_PERMUTATION_INDEX):03d}__prediction.png"
        )
    )
    synthetic_prediction_saved = True
    synthetic_prediction_status = "saved"
    synthetic_prediction_drug_name = "synthetic_example"

    try:
        synthetic_batch = load_synthetic_prediction_batch(
            datamodule,
            split=SYNTHETIC_SPLIT,
            batch_index=SYNTHETIC_BATCH_INDEX,
            permutation_index=SYNTHETIC_PERMUTATION_INDEX,
        )
        # synthetic_batch.target_obs     : [B, It, To, 1]
        # synthetic_batch.target_rem_sim : [B, It, Tr, 1]
        synthetic_prediction_drug_name = str(
            synthetic_batch.substance_name[0] if synthetic_batch.substance_name else "synthetic_example"
        ).strip() or "synthetic_example"
        save_prediction_plot(
            model=model,
            batch=synthetic_batch,
            image_path=synthetic_prediction_path,
            device=device,
            sample_size=PREDICTION_SAMPLE_SIZE,
            log_scale=PREDICTION_LOG_SCALE,
            drug_name=synthetic_prediction_drug_name,
            plot_kwargs=predictive_plot_kwargs,
        )
    except Exception as exc:
        synthetic_prediction_saved = False
        synthetic_prediction_status = f"failed: {exc}"
        synthetic_prediction_path = None

    synthetic_prediction_summary = pd.DataFrame.from_records(
        [
            {
                "split": SYNTHETIC_SPLIT,
                "batch_index": SYNTHETIC_BATCH_INDEX,
                "permutation_index": SYNTHETIC_PERMUTATION_INDEX,
                "drug_name": synthetic_prediction_drug_name,
                "prediction_saved": synthetic_prediction_saved,
                "prediction_status": synthetic_prediction_status,
                "prediction_path": (
                    str(synthetic_prediction_path)
                    if synthetic_prediction_path is not None
                    else None
                ),
            }
        ]
    )

synthetic_prediction_summary
